# Test Suite: Model Pipeline & Evaluation Module
This notebook executes the unit tests and interactive validations for `src/evaluate.py`.
It tests:
- Evaluation metrics extraction (`evaluate_classifier`)
- Precision, recall, macro F1, confusion matrix, ROC-AUC, and latency computation
- Dynamic decision threshold tuning (`compute_threshold_metrics`) across variable thresholds


In [1]:
import os
import sys
import unittest
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Working Directory: {os.getcwd()}")
from src.evaluate import evaluate_classifier, compute_threshold_metrics
print("Evaluation module imported successfully!")


Working Directory: e:\ML\mini project 2
Evaluation module imported successfully!


## Interactive Demonstration 1: Fit Mini-Pipeline & Evaluate


In [2]:
texts_train = [
    "This is a genuine product, works nicely.",
    "Loved it! High quality and sturdy.",
    "Worst fake garbage, broke immediately.",
    "Scam product, do not buy."
]
y_train = np.array([0, 0, 1, 1])

texts_test = [
    "Good product, fine quality.",
    "Fake and defective."
]
y_test = np.array([0, 1])

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression())
])
pipeline.fit(texts_train, y_train)

metrics = evaluate_classifier(pipeline, texts_test, y_test, model_name="Mini_LR_Model", train_time_sec=0.05)
for k, v in metrics.items():
    print(f"  {k:20s}: {v}")


  model_name          : Mini_LR_Model
  accuracy            : 1.0
  precision_fake      : 1.0
  recall_fake         : 1.0
  f1_fake             : 1.0
  precision_honest    : 1.0
  recall_honest       : 1.0
  f1_honest           : 1.0
  macro_f1            : 1.0
  weighted_f1         : 1.0
  roc_auc             : 1.0
  pr_auc              : 1.0
  confusion_matrix    : {'tn': 1, 'fp': 0, 'fn': 0, 'tp': 1}
  train_time_sec      : 0.05
  pred_latency_ms_per_100: 37.19
  model_size_mb       : 0.0
  y_pred              : [0 1]
  y_scores            : [0.47001479 0.50276725]


## Interactive Demonstration 2: Dynamic Decision Threshold Tuning


In [3]:
y_true = np.array([0, 1])
y_probs = np.array([0.2, 0.8])
thresholds = np.array([0.3, 0.5, 0.7])

df_thresh = compute_threshold_metrics(y_true, y_probs, thresholds=thresholds)
display(df_thresh)


,threshold,accuracy,precision_fake,recall_fake,f1_fake,macro_f1
0,0.3,1.0,1.0,1.0,1.0,1.0
1,0.5,1.0,1.0,1.0,1.0,1.0
2,0.7,1.0,1.0,1.0,1.0,1.0


## Formal Unittest Suite Runner


In [4]:
class TestModelPipeline(unittest.TestCase):

    def setUp(self):
        self.texts_train = [
            "This is a genuine product, works nicely.",
            "Loved it! High quality and sturdy.",
            "Worst fake garbage, broke immediately.",
            "Scam product, do not buy."
        ]
        self.y_train = np.array([0, 0, 1, 1])

        self.texts_test = [
            "Good product, fine quality.",
            "Fake and defective."
        ]
        self.y_test = np.array([0, 1])

        self.pipeline = Pipeline([
            ("tfidf", TfidfVectorizer()),
            ("clf", LogisticRegression())
        ])
        self.pipeline.fit(self.texts_train, self.y_train)

    def test_evaluate_classifier(self):
        metrics = evaluate_classifier(
            self.pipeline,
            self.texts_test,
            self.y_test,
            model_name="Test_LogisticRegression",
            train_time_sec=0.15
        )
        self.assertIn("accuracy", metrics)
        self.assertIn("macro_f1", metrics)
        self.assertIn("precision_fake", metrics)
        self.assertIn("recall_fake", metrics)
        self.assertIn("confusion_matrix", metrics)
        self.assertIn("model_size_mb", metrics)
        self.assertEqual(metrics["model_name"], "Test_LogisticRegression")

    def test_threshold_metrics(self):
        y_scores = np.array([0.2, 0.8])
        th_df = compute_threshold_metrics(self.y_test, y_scores, thresholds=np.array([0.3, 0.5, 0.7]))
        self.assertEqual(len(th_df), 3)
        self.assertIn("precision_fake", th_df.columns)
        self.assertIn("recall_fake", th_df.columns)

# Run unittest suite in notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestModelPipeline)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Some tests failed!"


test_evaluate_classifier (__main__.TestModelPipeline.test_evaluate_classifier) ... ok
test_threshold_metrics (__main__.TestModelPipeline.test_threshold_metrics) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.041s

OK
